In [1]:
import blocks as layers
import torch
import torch.nn as nn
import math

/home/cybertesla/Transformers/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [6]:
class Transformer(nn.Module):

    def __init__(self, src_vocab_size: int, tgt_vocab_size: int, d_model: int, num_heads: int, hidden_dim: int, num_blocks: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.dropout_rate = dropout_rate
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = layers.PositionalEncoding()
        self.dropout = nn.Dropout(p=dropout_rate)
        self.encoder = layers.EncoderStack(d_model, num_heads, hidden_dim, num_blocks, dropout_rate)
        self.decoder = layers.DecoderStack(d_model, num_heads, hidden_dim, num_blocks, dropout_rate)
        self.final_linear_layer = nn.Linear(d_model, tgt_vocab_size)


    def forward(self, src_tokens: torch.Tensor, tgt_tokens: torch.Tensor):
        # src_tokens, tgt_tokens Dimension: (Batch, Tokens)
        
        encoder_embeddings = self.encoder_embedding(src_tokens) * math.sqrt(self.d_model)
        decoder_embeddings = self.decoder_embedding(tgt_tokens) * math.sqrt(self.d_model)
        # Dimension: (Batch_size, num_tokens, d_model)
        
        encoder_positional_embeddings = self.dropout(self.positional_encoding(encoder_embeddings))
        decoder_positional_embeddings = self.dropout(self.positional_encoding(decoder_embeddings))
        # Dimension: (Batch_size, num_tokens, d_model)

        # Passing through Encoder
        encoder_output = self.encoder(encoder_positional_embeddings)

        # Passing through Decoder
        decoder_output = self.decoder(decoder_positional_embeddings, encoder_output)

        # Final Linear Layer
        logits = self.final_linear_layer(decoder_output)

        return logits

In [7]:
batch_size = 2
num_src_tokens = 3
num_tgt_tokens = 5
d_model = 8
num_heads = 2
head_dim = 4
src_vocab_size = 5
tgt_vocab_size = 7
hidden_dim = 16
num_blocks = 6

In [8]:
dummy_src_input = torch.randint(low=0, high=src_vocab_size, size=(batch_size, num_src_tokens))
dummy_tgt_input = torch.randint(low=0, high=tgt_vocab_size, size=(batch_size, num_tgt_tokens))
print(dummy_src_input.shape)
print(dummy_tgt_input.shape)
print("(batch_size, num_tokens)")

torch.Size([2, 3])
torch.Size([2, 5])
(batch_size, num_tokens)


In [9]:
transformer =Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, hidden_dim, num_blocks)
transformer

Transformer(
  (encoder_embedding): Embedding(5, 8)
  (decoder_embedding): Embedding(7, 8)
  (positional_encoding): PositionalEncoding()
  (dropout): Dropout(p=0.1, inplace=False)
  (encoder): EncoderStack(
    (layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (attention_layer): MultiHeadAttention(
          (w_q): Linear(in_features=8, out_features=8, bias=False)
          (w_k): Linear(in_features=8, out_features=8, bias=False)
          (w_v): Linear(in_features=8, out_features=8, bias=False)
          (w_o): Linear(in_features=8, out_features=8, bias=False)
        )
        (ffnn): FFNN(
          (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
          (output_layer): Linear(in_features=16, out_features=8, bias=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
       

In [10]:
transformer_output = transformer(dummy_src_input, dummy_tgt_input)
print(transformer_output.shape)
print("(batch_size, num_tgt_tokens, tgt_vocab_size)")
print(transformer_output)

torch.Size([2, 5, 7])
(batch_size, num_tgt_tokens, tgt_vocab_size)
tensor([[[ 0.3862, -0.0659,  0.6718, -0.6070, -0.0979, -0.6033,  0.2416],
         [ 1.1976,  0.1460, -0.0629,  0.2834,  0.0864, -0.3950,  0.2450],
         [ 1.2429,  0.2281,  0.5717,  0.1855, -0.2464, -1.2010,  0.0838],
         [ 1.0213,  0.3114,  1.4492,  0.0474, -0.6773, -0.9140, -0.2585],
         [ 0.4228,  0.5342,  0.6603, -0.3767, -1.1632, -0.6214, -1.2850]],

        [[ 1.0101,  0.0918,  0.0791,  0.2659,  0.3833, -0.2040,  0.5134],
         [ 0.9324,  0.5465,  0.4577, -0.4244, -0.8020, -1.0748, -0.9165],
         [ 1.1989,  0.0923, -0.1439,  0.3577,  0.2170, -0.3381,  0.3840],
         [ 0.6455,  0.6403,  0.8409, -0.5038, -1.1661, -0.9429, -1.3521],
         [ 0.4117,  0.6265,  0.9301, -0.5268, -1.0658, -0.7524, -1.2759]]],
       grad_fn=<ViewBackward0>)
